# Functional Data Analysis with R and MATLAB: the figures in fdatools

This notebook rebuilds the figures of J. O. Ramsay, G. Hooker and S. Graves
(2009), *Functional Data Analysis with R and MATLAB*, Springer (Use R! series),
with the public API of fdatools only. It is generated by
`tools/build_book_notebook.py` from the chapter files in `notebooks/book/`;
edit those files, not this notebook.

**Data.** `growth`, `gait` and `pinch` ship inside the package. The other book
datasets are downloaded on first use from the `data-v1` GitHub release of the
project, checked against a SHA-256 checksum and cached in `~/.cache/fdatools`. Set
the environment variable `FDATOOLS_DATA_DIR` to use another cache directory, or to
point at a local copy of the release files (for example the `data_release/`
directory written by `tools/build_data_release.py`). Where the book uses data
that fdatools does not ship, the figure's text says what was used instead.

**Run it.** `pytest --nbmake notebooks/book_figures.ipynb` executes every cell;
each chapter file also runs on its own with
`MPLBACKEND=Agg python notebooks/book/chNN_<name>.py`.

# Chapter 1: Introduction to functional data analysis

Figures 1.1 to 1.10 of Ramsay, Hooker & Graves (2009). The chapter shows the
kinds of data the book works with: growth curves, an economic index, refinery
input/output records, gait cycles, handwriting and weather records.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

### Figure 1.1
The heights of the first 10 girls of the Berkeley growth study at 31 ages
(circles), with the monotone smooth of each record (lines).

In [ ]:
growth = fdt.datasets.load_growth()
age = growth.age
heights = growth.hgtf[:, :10]
growth_basis = fdt.BSpline(domain=(1.0, 18.0), breaks=age, order=6)
girls = fdt.smooth(heights, age, basis=growth_basis, lam=0.01, penalty=3, constraint="monotone")
fine_age = np.linspace(1.0, 18.0, 401)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fine_age, girls(fine_age), color="0.3", linewidth=1.0)
ax.plot(age, heights, "o", color="k", markersize=3, fillstyle="none")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Height (cm)")
fig

### Figure 1.2
The estimated accelerations of height of the same 10 girls (cm/yr^2); the heavy
dashed line is their mean. The pubertal growth spurt is the dip near age 11-13.
A monotone fit is x = b0 + b1 * integral of exp(W), so x'' = x' * W'.

In [ ]:
acceleration = girls(fine_age, deriv=1) * girls.fd(fine_age, deriv=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fine_age, acceleration, color="0.5", linewidth=0.8)
ax.plot(fine_age, acceleration.mean(axis=1), "k--", linewidth=2.5)
ax.axhline(0.0, color="k", linestyle=":", linewidth=0.8)
ax.set_xlim(3.0, 18.0)
ax.set_ylim(-4.0, 2.0)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Acceleration (cm/yr$^2$)")
fig

### Figure 1.3
The monthly index of nondurable goods manufacturing in the United States.
Dates follow the dataset metadata: monthly values starting January 1919.

In [ ]:
nondurables = fdt.datasets.load_nondurables()
index = np.asarray(nondurables.value, dtype=float)
year = 1919.0 + np.arange(index.size) / 12.0

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(year, index, color="k", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Nondurable goods index")
fig

### Figure 1.4
Refinery data: the amount of product on tray 47 of a distillation column (top)
and the reflux flow into that tray (bottom) during an experiment.

In [ ]:
refinery = fdt.datasets.load_refinery()

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
top.plot(refinery.time, refinery.tray47, "o", color="k", markersize=2.5)
top.set_ylabel("Tray 47 level")
bottom.plot(refinery.time, refinery.reflux, "o", color="k", markersize=2.5)
bottom.set_ylabel("Reflux flow")
bottom.set_xlabel("Time (min)")
fig.tight_layout()
fig

### Figure 1.5
Hip and knee angles in the sagittal plane for 39 children over one gait cycle
(time is the proportion of the cycle). Curves are Fourier smooths of the data.

In [ ]:
gait = fdt.datasets.load_gait()
gait_basis = fdt.Fourier(domain=(0.0, 1.0), n_basis=21)
gait_fd = fdt.smooth(
    np.asarray(gait.value), gait.t, basis=gait_basis, lam=1e-11, penalty=fdt.LDO.harmonic(1.0)
).fd
cycle = np.linspace(0.0, 1.0, 201)
gait_values = gait_fd(cycle)

fig, (hip_ax, knee_ax) = plt.subplots(1, 2, figsize=(10, 4.5))
hip_ax.plot(cycle, gait_values[:, :, 0], linewidth=0.7)
hip_ax.set_title("Hip")
hip_ax.set_ylabel("Hip angle (degrees)")
knee_ax.plot(cycle, gait_values[:, :, 1], linewidth=0.7)
knee_ax.set_title("Knee")
knee_ax.set_ylabel("Knee angle (degrees)")
for axis in (hip_ax, knee_ax):
    axis.set_xlabel("Time (proportion of gait cycle)")
fig.tight_layout()
fig

### Figure 1.6
Knee angle against hip angle over one cycle for one child (solid) and for the
mean child (dashed). Letters A-E mark five equally spaced moments of the cycle.

In [ ]:
marks = np.linspace(0.0, 0.8, 5)
child = gait_fd[4](cycle)[:, 0, :]
average = gait_fd.mean()(cycle)[:, 0, :]
child_marks = gait_fd[4](marks)[:, 0, :]
mean_marks = gait_fd.mean()(marks)[:, 0, :]

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(child[:, 0], child[:, 1], "k-", linewidth=1.2)
ax.plot(average[:, 0], average[:, 1], "k--", linewidth=1.2)
for letter, point, centre in zip("ABCDE", child_marks, mean_marks, strict=True):
    ax.text(point[0], point[1], letter, fontsize=12, fontweight="bold")
    ax.text(centre[0], centre[1], letter, fontsize=10, color="0.4")
ax.set_xlabel("Hip angle (degrees)")
ax.set_ylabel("Knee angle (degrees)")
fig

### Figure 1.7
Twenty samples of the script "fda" written by one person: X against Y
coordinates of the pen tip (meters), each record centered.

In [ ]:
handwriting = fdt.datasets.load_handwriting()
pen = np.asarray(handwriting.value)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(pen[:, :, 0], pen[:, :, 1], color="k", linewidth=0.5)
ax.set_aspect("equal")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
fig

### Figure 1.8
Mean daily temperature at four Canadian weather stations: Montreal, Edmonton,
Prince Rupert and Resolute, smoothed with a 65-term Fourier basis.

In [ ]:
weather = fdt.datasets.load_canadian_weather()
chosen = ["Montreal", "Edmonton", "Pr. Rupert", "Resolute"]
columns = [weather.stations.index(name) for name in chosen]
day = np.asarray(weather.t, dtype=float) - 0.5
temp_fd = fdt.smooth(
    weather.temp[:, columns],
    day,
    basis=fdt.Fourier(domain=(0.0, 365.0), n_basis=65),
    lam=1e4,
    penalty=fdt.LDO.harmonic(365.0),
).fd
year_grid = np.linspace(0.0, 365.0, 366)

fig, ax = plt.subplots(figsize=(8, 5))
for name, style, curve in zip(chosen, ["-", "--", "-.", ":"], temp_fd(year_grid).T, strict=True):
    ax.plot(year_grid, curve, "k", linestyle=style, label=name)
ax.set_xlabel("Day")
ax.set_ylabel("Mean temperature (deg C)")
ax.legend(loc="lower center")
fig

### Figure 1.9
The log10 nondurable goods index for 1964-1967, a period of steady growth
(circles), with a smooth fit (solid) and the straight-line trend (dashed).

In [ ]:
log_index = np.log10(index)
window = (year >= 1962.0) & (year < 1970.0)
window_year = year[window]
window_log = log_index[window]
monthly_breaks = np.linspace(1962.0, 1970.0, 97)
index_fd = fdt.smooth(
    window_log,
    window_year,
    basis=fdt.BSpline(domain=(1962.0, 1970.0), breaks=monthly_breaks, order=6),
    lam=1e-6,
    penalty=4,
).fd
shown = (window_year >= 1964.0) & (window_year < 1968.0)
fine_year = np.linspace(1964.0, 1968.0, 481)
slope, intercept = np.polyfit(window_year[shown], window_log[shown], 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(window_year[shown], window_log[shown], "o", color="k", markersize=3, fillstyle="none")
ax.plot(fine_year, index_fd(fine_year)[:, 0], "k-", linewidth=1.0)
ax.plot(fine_year, intercept + slope * fine_year, "k--", linewidth=1.0)
ax.set_xlabel("Year")
ax.set_ylabel("log10 nondurable goods index")
fig

### Figure 1.10
Phase-plane plot for 1964: acceleration against velocity of the smoothed log10
nondurable goods index, labelled by month. Seasonal cycles make loops.

In [ ]:
months = "JFMAMJJASOND"
month_labels = {1964.0 + (m + 0.5) / 12.0: months[m] for m in range(12)}
year_1964 = np.linspace(1964.0, 1965.0, 201)

fig, ax = plt.subplots(figsize=(6, 6))
fdt.phase_plane(index_fd, year_1964, labels=month_labels, ax=ax, color="k")
ax.axhline(0.0, color="k", linestyle=":", linewidth=0.8)
ax.axvline(0.0, color="k", linestyle=":", linewidth=0.8)
ax.set_xlabel("Velocity")
ax.set_ylabel("Acceleration")
fig

# Chapter 3: How to specify basis systems for building functions

Figures 3.1 to 3.4 of Ramsay, Hooker & Graves (2009). A function is a weighted
sum of basis functions; this chapter looks at B-spline bases: how a spline is
built from its basis, how the order controls smoothness, and how well a spline
basis approximates a known function and its derivatives.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

### Figure 3.1
A cubic (order 4) spline function on [0, 10] with nine equally spaced interior
knots (dotted lines). The coefficients are a fixed smooth sequence chosen for
illustration, as in the book, where the spline is built from its coefficients.

In [ ]:
spline_basis = fdt.BSpline(domain=(0.0, 10.0), n_basis=13, order=4)
knot_points = np.asarray(spline_basis.breaks)
spline_coefs = np.array([0.0, 1.5, 2.8, 1.2, -1.0, -2.2, -0.5, 1.8, 2.6, 1.0, -0.8, -1.5, 0.2])
spline_fd = fdt.FData(spline_coefs, spline_basis)
x_grid = np.linspace(0.0, 10.0, 501)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x_grid, spline_fd(x_grid)[:, 0], "k-", linewidth=1.5)
for knot in knot_points[1:-1]:
    ax.axvline(knot, color="0.5", linestyle=":", linewidth=0.8)
ax.set_xlabel("t")
ax.set_ylabel("x(t)")
fig

### Figure 3.2
The thirteen B-spline basis functions of order 4 with nine equally spaced
interior knots (dashed lines) on [0, 10].

In [ ]:
phi = spline_basis(x_grid)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x_grid, phi, "k-", linewidth=1.0)
for knot in knot_points[1:-1]:
    ax.axvline(knot, color="0.5", linestyle="--", linewidth=0.8)
ax.set_xlabel("t")
ax.set_ylabel("B-spline basis functions")
fig

### Figure 3.3
B-spline bases of orders 1 (step functions) to 4 (cubic) on the same breaks.
Each order adds one continuous derivative at the knots. fdatools requires distinct
breaks, so the book's view of smoothness at a knot is shown through the order.

In [ ]:
breaks = np.linspace(0.0, 1.0, 5)
unit_grid = np.linspace(0.0, 1.0, 401)

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for order, axis in zip(range(1, 5), axes.ravel(), strict=True):
    basis = fdt.BSpline(domain=(0.0, 1.0), breaks=breaks, order=order)
    axis.plot(unit_grid, basis(unit_grid), linewidth=1.0)
    for knot in breaks[1:-1]:
        axis.axvline(knot, color="0.5", linestyle=":", linewidth=0.8)
    axis.set_title(f"Order {order}: {basis.n_basis} basis functions")
for axis in axes[1]:
    axis.set_xlabel("t")
fig.tight_layout()
fig

### Figure 3.4
Approximating sin(t) on [0, 2 pi] by least squares with 13 order-4 B-splines:
the error of the approximation (top) and of its first (middle) and second
(bottom) derivatives. Errors grow with each derivative taken.

In [ ]:
circle = np.linspace(0.0, 2.0 * np.pi, 201)
sine_basis = fdt.BSpline(domain=(0.0, 2.0 * np.pi), n_basis=13, order=4)
sine_coefs, *_ = np.linalg.lstsq(sine_basis(circle), np.sin(circle), rcond=None)
sine_fd = fdt.FData(sine_coefs, sine_basis)
exact = [np.sin(circle), np.cos(circle), -np.sin(circle)]
titles = ["sin(t)", "D sin(t)", "D$^2$ sin(t)"]

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
for deriv, axis in enumerate(axes):
    error = sine_fd(circle, deriv=deriv)[:, 0] - exact[deriv]
    axis.plot(circle, error, "k-", linewidth=1.0)
    axis.axhline(0.0, color="0.5", linestyle=":", linewidth=0.8)
    axis.set_ylabel(f"Error in {titles[deriv]}")
axes[-1].set_xlabel("t")
fig.tight_layout()
fig

# Chapter 4: How to build functional data objects

Figures 4.1 to 4.3 of Ramsay, Hooker & Graves (2009). A functional data object
is a coefficient array plus a basis. This chapter builds such objects for the
Canadian weather data, by smoothing and by plain regression, and applies a
linear differential operator to them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

### Figure 4.1
Mean daily temperature curves for the 35 Canadian weather stations, estimated
by smoothing the daily data with a 65-term Fourier basis.

In [ ]:
weather = fdt.datasets.load_canadian_weather()
day = np.asarray(weather.t, dtype=float) - 0.5
temp_basis = fdt.Fourier(domain=(0.0, 365.0), n_basis=65)
harmonic = fdt.LDO.harmonic(365.0)
temp_fd = fdt.smooth(weather.temp, day, basis=temp_basis, lam=1e4, penalty=harmonic).fd

fig, ax = plt.subplots(figsize=(8, 5))
temp_fd.plot(ax=ax, color="k", linewidth=0.6)
ax.set_xlabel("Day (January 1 to December 31)")
ax.set_ylabel("Mean temperature (deg C)")
fig

### Figure 4.2
Montreal's daily mean temperatures (circles) and the curve fitted by ordinary
least-squares regression on a 13-term Fourier basis (no roughness penalty).

In [ ]:
montreal = weather.stations.index("Montreal")
regression_basis = fdt.Fourier(domain=(0.0, 365.0), n_basis=13)
design = regression_basis(day)
montreal_coefs, *_ = np.linalg.lstsq(design, weather.temp[:, montreal], rcond=None)
montreal_fd = fdt.FData(montreal_coefs, regression_basis)

fig, ax = plt.subplots(figsize=(8, 5))
montreal_fd.plot_fit(weather.temp[:, montreal], day, ax=ax, color="k", linewidth=1.5)
ax.set_xlabel("Day (January 1 to December 31)")
ax.set_ylabel("Temperature (deg C)")
ax.set_title("Montreal")
fig

### Figure 4.3
The harmonic acceleration L x = (2 pi / 365)^2 D x + D^3 x of the 35 smoothed
temperature curves. L removes pure sinusoids of period one year, so what is
left is the part of each curve that is not a simple annual cycle.

In [ ]:
year_grid = np.linspace(0.0, 365.0, 366)
accelerated = temp_fd(year_grid, harmonic)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(year_grid, accelerated, color="k", linewidth=0.6)
ax.axhline(0.0, color="0.5", linestyle=":", linewidth=0.8)
ax.set_xlabel("Day (January 1 to December 31)")
ax.set_ylabel("L-temperature")
fig

# Chapter 5: Smoothing: Computing Curves from Noisy Data

Figures 5.1 to 5.8 of Ramsay, Hooker and Graves (2009): choosing the smoothing
parameter by GCV, roughness-penalised smoothing of the growth and log
precipitation data, and constrained fits (positive, monotone, density).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

growth = fdt.datasets.load_growth()
weather = fdt.datasets.load_canadian_weather()
age = growth.age
hgtf = growth.hgtf
day5 = weather.t - 0.5  # the book's day.5: mid-day points 0.5, ..., 364.5
harmonic = fdt.LDO.harmonic(period=365.0)
daybasis = fdt.Fourier(domain=(0.0, 365.0), n_basis=365)

### Figure 5.1
GCV criterion against log10(lambda) for smoothing the 54 Berkeley girls' heights
with order-6 B-splines (knots at the ages) and a fourth-derivative penalty. The
criterion is summed over girls.

In [ ]:
growthbasis = fdt.BSpline(domain=(1.0, 18.0), order=6, breaks=age)
loglam = np.arange(-6.0, 0.01, 0.25)
gcvsave = np.array(
    [
        float(np.sum(fdt.smooth(hgtf, age, basis=growthbasis, lam=10.0**ll, penalty=4).gcv))
        for ll in loglam
    ]
)
best_loglam = float(loglam[np.argmin(gcvsave)])
print(f"GCV minimum at log10(lambda) = {best_loglam:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(loglam, gcvsave, "o-", color="black", markersize=4)
ax.set_xlabel(r"$\log_{10}(\lambda)$")
ax.set_ylabel(r"GCV($\lambda$)")
ax.set_title("GCV for the girls' growth data")
fig

### Figure 5.2
Acceleration (second derivative) curves of the first ten girls with lambda = 0.1
(the GCV minimum of Figure 5.1 fits the heights well but leaves the second
derivative rough); the heavy dashed line is the mean acceleration of all 54 girls.

In [ ]:
growthfit = fdt.smooth(hgtf, age, basis=growthbasis, lam=0.1, penalty=4)
print(f"growth: lambda = 0.1, df = {growthfit.df:.2f}")
agefine = np.linspace(1.0, 18.0, 401)
accel = np.asarray(growthfit.fd(agefine, deriv=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(agefine, accel[:, :10], linewidth=1)
ax.plot(agefine, accel.mean(axis=1), "k--", linewidth=3, label="mean (54 girls)")
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_ylim(-4.0, 2.0)
ax.set_xlabel("Age (years)")
ax.set_ylabel(r"Height acceleration (cm/yr$^2$)")
ax.legend()
fig

### Figure 5.3
Log10 daily precipitation at the 35 Canadian weather stations, smoothed with a
365-term Fourier basis and the harmonic acceleration penalty; lambda is chosen by
GCV over 1e4, ..., 1e9 (the book's choice, 1e6, is the GCV minimum).

In [ ]:
logprec = weather.log10precip
precgcv = np.array(
    [
        float(np.sum(fdt.smooth(logprec, day5, basis=daybasis, lam=10.0**ll, penalty=harmonic).gcv))
        for ll in range(4, 10)
    ]
)
prec_lam = 10.0 ** (4 + int(np.argmin(precgcv)))
logprecfit = fdt.smooth(logprec, day5, basis=daybasis, lam=prec_lam, penalty=harmonic)
print(f"log precipitation: lambda = {prec_lam:.0e}, df = {logprecfit.df:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
logprecfit.fd.plot(ax=ax, linewidth=1)
ax.set_xlabel("Day (July 1 to June 30)")
ax.set_ylabel(r"$\log_{10}$ precipitation (mm)")
ax.set_title("Smoothed log precipitation, 35 stations")
fig

### Figure 5.4
Vancouver's average daily precipitation with a positive smooth x(t) = exp W(t),
W expanded in the 365-term Fourier basis with a harmonic acceleration penalty.

In [ ]:
vancouver = weather.stations.index("Vancouver")
vanprec = weather.precip[:, vancouver]
vanfit = fdt.smooth(vanprec, day5, basis=daybasis, lam=1e4, penalty=harmonic, constraint="positive")
print(f"Vancouver positive smooth: lambda = 1e4, df = {vanfit.df:.2f}")
dayfine = np.linspace(0.0, 365.0, 731)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(day5, vanprec, "o", markersize=2, fillstyle="none", color="grey", label="data")
ax.plot(dayfine, np.asarray(vanfit(dayfine)), "k-", linewidth=2, label="positive smooth")
ax.set_xlabel("Day (July 1 to June 30)")
ax.set_ylabel("Precipitation (mm)")
ax.set_title("Vancouver precipitation")
ax.legend()
fig

### Figure 5.5
Tibia lengths of one newborn infant over its first 40 days, with a monotone smooth
x(t) = b0 + b1 int exp W (order-6 B-splines, knots at every day, D^3 penalty on W).

In [ ]:
infant = fdt.datasets.load_infant_growth()
tibiabasis = fdt.BSpline(domain=(1.0, 40.0), order=6, breaks=infant.day)
tibiafit = fdt.smooth(
    infant.tibia_length,
    infant.day,
    basis=tibiabasis,
    lam=1e-1,
    penalty=3,
    constraint="monotone",
)
dayinf = np.linspace(1.0, 40.0, 391)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(infant.day, infant.tibia_length, "o", fillstyle="none", color="grey", label="data")
ax.plot(dayinf, np.asarray(tibiafit(dayinf)), "k-", linewidth=2, label="monotone smooth")
ax.set_xlabel("Day")
ax.set_ylabel("Tibia length (mm)")
ax.legend()
fig

### Figure 5.6
Velocity of tibia growth from the monotone fit of Figure 5.5; the fit's derivative
b1 exp W(t) is positive everywhere, showing the growth spurts.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(dayinf, np.asarray(tibiafit(dayinf, deriv=1)), "k-", linewidth=2)
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Day")
ax.set_ylabel("Tibia velocity (mm/day)")
fig

### Figure 5.7
Height velocity of the first ten Berkeley girls from monotone smoothing (knots at
the ages, order 6, D^3 penalty on W): unlike an unconstrained fit, the fitted
heights can never decrease.

In [ ]:
girlsfit = fdt.smooth(
    hgtf[:, :10], age, basis=growthbasis, lam=1e-1, penalty=3, constraint="monotone"
)
velocity = np.asarray(girlsfit(agefine, deriv=1))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(agefine, velocity, linewidth=1)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Height velocity (cm/yr)")
ax.set_title("Monotone smooth, first ten girls")
fig

### Figure 5.8
Estimated density of June daily precipitation in Regina (days with 2 to 45 mm),
p(x) = exp W(x) / int exp W. fdatools' smooth() has no density constraint, so the
penalised log-likelihood is maximised here from the public Basis primitives.

In [ ]:
regina = fdt.datasets.load_regina_precip().value
rain = np.sort(regina[(regina > 2.0) & (regina <= 45.0)])
densbasis = fdt.BSpline(domain=(2.0, 45.0), n_basis=13)
densgrid = np.linspace(2.0, 45.0, 861)
phi_data = np.asarray(densbasis(rain))
phi_grid = np.asarray(densbasis(densgrid))
penmat = np.asarray(densbasis.penalty(2))
quad = np.full(densgrid.size, densgrid[1] - densgrid[0])
quad[[0, -1]] *= 0.5
dens_lam = 1e-1


def neg_loglik(coefs):
    w = phi_grid @ coefs
    shift = w.max()
    mass = np.exp(w - shift)
    total = quad @ mass
    value = -phi_data.sum(axis=0) @ coefs + rain.size * (np.log(total) + shift)
    value += dens_lam * coefs @ penmat @ coefs
    grad = -phi_data.sum(axis=0) + rain.size * (phi_grid.T @ (quad * mass)) / total
    return value, grad + 2.0 * dens_lam * penmat @ coefs


opt = minimize(neg_loglik, np.zeros(densbasis.n_basis), jac=True, method="BFGS")
wgrid = phi_grid @ opt.x
density = np.exp(wgrid - wgrid.max())
density /= quad @ density

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(rain, bins=30, density=True, color="lightgrey", edgecolor="grey")
ax.plot(densgrid, density, "k-", linewidth=2)
ax.plot(rain, np.zeros_like(rain), "|", color="black", markersize=8)
ax.set_xlabel("Precipitation (mm)")
ax.set_ylabel("Probability density")
ax.set_title(f"Regina June precipitation ({rain.size} days)")
fig

# Chapter 6: Descriptions of Functional Data

Figures 6.1 to 6.8 of Ramsay, Hooker and Graves (2009): covariance and
correlation surfaces, the residual variance, phase-plane plots (a sinusoid, the
nondurable goods index, girls' growth) and pointwise confidence limits.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

weather = fdt.datasets.load_canadian_weather()
day5 = weather.t - 0.5  # the book's day.5: mid-day points 0.5, ..., 364.5
harmonic = fdt.LDO.harmonic(period=365.0)
daybasis = fdt.Fourier(domain=(0.0, 365.0), n_basis=365)
logprec = weather.log10precip
logprecfit = fdt.smooth(logprec, day5, basis=daybasis, lam=1e6, penalty=harmonic)
logprecfd = logprecfit.fd
tempfd = fdt.smooth(weather.temp, day5, basis=daybasis, lam=1e4, penalty=harmonic).fd
dayfine = np.linspace(0.0, 365.0, 121)

### Figure 6.1
The variance-covariance surface v(s, t) of the smoothed log precipitation curves,
as a perspective plot (left) and a contour plot (right).

In [ ]:
logprecvar = fdt.stats.cov(logprecfd)
surface = np.asarray(logprecvar(dayfine, dayfine))
grid_s, grid_t = np.meshgrid(dayfine, dayfine, indexing="ij")

fig = plt.figure(figsize=(11, 4.5))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(grid_s, grid_t, surface, cmap="viridis", linewidth=0)
ax3d.set_xlabel("Day (s)")
ax3d.set_ylabel("Day (t)")
ax3d.set_zlabel("v(s, t)")
ax2 = fig.add_subplot(1, 2, 2)
contours = ax2.contour(dayfine, dayfine, surface.T, levels=12, cmap="viridis")
ax2.clabel(contours, fontsize=7)
ax2.set_xlabel("Day (s)")
ax2.set_ylabel("Day (t)")
ax2.set_aspect("equal")
fig.suptitle("Variance-covariance surface of log precipitation")
fig

### Figure 6.2
Cross-correlation corr(temperature(s), log precipitation(t)) across the 35
stations, from smoothed temperature and log precipitation curves.

In [ ]:
crosscor = np.asarray(fdt.stats.cor(tempfd, logprecfd, s=dayfine, t=dayfine))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
filled = ax.contourf(dayfine, dayfine, crosscor.T, levels=np.linspace(-1, 1, 21), cmap="RdBu_r")
ax.contour(dayfine, dayfine, crosscor.T, levels=[0.0], colors="black", linewidths=1)
fig.colorbar(filled, ax=ax, label="correlation")
ax.set_xlabel("Temperature day (s)")
ax.set_ylabel("Log precipitation day (t)")
fig

### Figure 6.3
Standard deviation of the log precipitation residuals across stations for each
day (points) with its smooth, the diagonal of the residual covariance Sigma_e.
The log variance is smoothed so the fitted standard deviation stays positive.

In [ ]:
logprecres = logprec - np.asarray(logprecfd(day5))
logprecvar1 = np.sum(logprecres**2, axis=1) / (logprec.shape[1] - 1)
logvarfit = fdt.smooth(
    np.log(logprecvar1),
    day5,
    basis=fdt.Fourier(domain=(0.0, 365.0), n_basis=365),
    penalty=harmonic,
)
print(f"log variance smooth: GCV lambda = {logvarfit.lam:.3g}, df = {logvarfit.df:.2f}")
sd_smooth = np.exp(np.asarray(logvarfit.fd(day5)) / 2.0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(day5, np.sqrt(logprecvar1), "o", markersize=2, fillstyle="none", color="grey")
ax.plot(day5, sd_smooth, "k-", linewidth=2)
ax.set_xlabel("Day (July 1 to June 30)")
ax.set_ylabel("Standard deviation of log10 precipitation")
fig

### Figure 6.4
Phase-plane plot of the simple harmonic x(t) = sin(2 pi t): acceleration against
velocity traces an ellipse; kinetic energy peaks on the horizontal axis and
potential energy on the vertical axis.

In [ ]:
unitbasis = fdt.Fourier(domain=(0.0, 1.0), n_basis=3)
sinscale = 1.0 / float(np.asarray(unitbasis(np.array([0.25])))[0, 1])
sinefd = fdt.FData(np.array([[0.0], [sinscale], [0.0]]), unitbasis)

fig, ax = plt.subplots(figsize=(5, 5))
fdt.phase_plane(
    sinefd,
    np.linspace(0.0, 1.0, 201),
    labels={0.0: "t=0", 0.25: "t=1/4", 0.5: "t=1/2", 0.75: "t=3/4"},
    ax=ax,
    color="black",
)
ax.axhline(0.0, color="grey", linestyle=":")
ax.axvline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Velocity")
ax.set_ylabel("Acceleration")
ax.set_title(r"Phase plane of $\sin(2\pi t)$")
fig

### Figure 6.5
Log10 of the US nondurable goods index, 1960 to 1970, with a smooth by order-6
B-splines with a knot at every month and a D^4 penalty.

In [ ]:
nondur = fdt.datasets.load_nondurables()
years = 1919.0 + np.arange(nondur.value.size) / 12.0
window = (years >= 1960.0) & (years < 1971.0)
ndyears = years[window]
lognondur = np.log10(nondur.value[window])
ndbasis = fdt.BSpline(domain=(ndyears[0], ndyears[-1]), order=6, breaks=ndyears)
ndfit = fdt.smooth(lognondur, ndyears, basis=ndbasis, lam=1e-7, penalty=4)
ndfine = np.linspace(ndyears[0], ndyears[-1], 1201)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ndyears, lognondur, "o", markersize=2, fillstyle="none", color="grey")
ax.plot(ndfine, np.asarray(ndfit.fd(ndfine)), "k-", linewidth=1.5)
ax.set_xlabel("Year")
ax.set_ylabel(r"$\log_{10}$ nondurable goods index")
fig

### Figure 6.6
Phase-plane plot of the nondurable goods index for 1964: acceleration against
velocity of the smooth in Figure 6.5, labelled with the months.

In [ ]:
months = "JFMAMJJASOND"
year64 = np.linspace(1964.0, 1965.0, 241)

fig, ax = plt.subplots(figsize=(5.5, 5))
fdt.phase_plane(
    ndfit.fd,
    year64,
    labels={1964.0 + (m + 0.5) / 12.0: months[m] for m in range(12)},
    ax=ax,
    color="black",
)
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Velocity")
ax.set_ylabel("Acceleration")
ax.set_title("Nondurable goods index, 1964")
fig

### Figure 6.7
Phase-plane plots of height acceleration against velocity for the first ten
Berkeley girls from age 3 to 18 (order-6 B-splines, D^4 penalty, lambda = 1); the
pubertal spurt is the large loop, and age 11.5 is marked on each curve.

In [ ]:
growth = fdt.datasets.load_growth()
growthbasis = fdt.BSpline(domain=(1.0, 18.0), order=6, breaks=growth.age)
girlsfd = fdt.smooth(growth.hgtf, growth.age, basis=growthbasis, lam=1.0, penalty=4).fd[:10]

fig, ax = plt.subplots(figsize=(6, 5))
fdt.phase_plane(girlsfd, np.linspace(3.0, 18.0, 301), labels={11.5: "o"}, ax=ax, linewidth=1)
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Velocity (cm/yr)")
ax.set_ylabel(r"Acceleration (cm/yr$^2$)")
fig

### Figure 6.8
Log precipitation at Prince Rupert with its smooth and 95% pointwise confidence
limits, from the linear data-to-coefficient map and the smoothed residual
variance of Figure 6.3 (Sigma_e taken as diagonal).

In [ ]:
rupert = weather.stations.index("Pr. Rupert")
sigma_e = np.exp(np.asarray(logvarfit.fd(day5)))
y2c = np.asarray(logprecfit.y2c_map)
phi = np.asarray(daybasis(day5))
coef_var = (y2c * sigma_e) @ y2c.T
fit_se = np.sqrt(np.einsum("ij,jk,ik->i", phi, coef_var, phi))
rupertfit = np.asarray(logprecfd(day5))[:, rupert]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(day5, logprec[:, rupert], "o", markersize=2, fillstyle="none", color="grey")
ax.plot(day5, rupertfit, "k-", linewidth=2, label="smooth")
ax.plot(day5, rupertfit + 2.0 * fit_se, "k--", linewidth=1, label="95% limits")
ax.plot(day5, rupertfit - 2.0 * fit_se, "k--", linewidth=1)
ax.set_xlabel("Day (July 1 to June 30)")
ax.set_ylabel(r"$\log_{10}$ precipitation (mm)")
ax.set_title("Prince Rupert")
ax.legend()
fig

# Chapter 7: Exploring Variation: Functional Principal and Canonical Components Analysis

Figures 7.1 to 7.6 of Ramsay, Hooker and Graves (2009): principal components of
the log precipitation curves (unrotated and varimax rotated) and their scores,
the bivariate PCA of the handwriting data, and the canonical correlation analysis
of temperature and log precipitation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0

weather = fdt.datasets.load_canadian_weather()
day5 = weather.t - 0.5  # the book's day.5: mid-day points 0.5, ..., 364.5
harmonic = fdt.LDO.harmonic(period=365.0)
daybasis = fdt.Fourier(domain=(0.0, 365.0), n_basis=365)
logprecfd = fdt.smooth(weather.log10precip, day5, basis=daybasis, lam=1e6, penalty=harmonic).fd
tempfd = fdt.smooth(weather.temp, day5, basis=daybasis, lam=1e4, penalty=harmonic).fd
stations = weather.stations

### Figure 7.1
The first four principal components of the log precipitation curves, each shown as
the mean curve (solid) plus (dashed) and minus (dotted) a multiple of the harmonic.

In [ ]:
logprecpca = fdt.FPCA(n=4).fit(logprecfd)
print("proportion of variance:", np.round(np.asarray(logprecpca.varprop), 4))

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
logprecpca.plot(ax=axes)
for ax in axes[1]:
    ax.set_xlabel("Day (July 1 to June 30)")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\log_{10}$ precipitation")
fig.tight_layout()
fig

### Figure 7.2
The four varimax-rotated principal components of the log precipitation curves,
again as perturbations of the mean; rotation concentrates each component on a
season.

In [ ]:
logprecrot = logprecpca.rotate("varimax")
print("rotated proportion of variance:", np.round(np.asarray(logprecrot.varprop), 4))

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
logprecrot.plot(ax=axes)
for ax in axes[1]:
    ax.set_xlabel("Day (July 1 to June 30)")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\log_{10}$ precipitation")
fig.tight_layout()
fig

### Figure 7.3
Scores of the 35 weather stations on the first two varimax-rotated components of
log precipitation, labelled by station.

In [ ]:
rotscores = np.asarray(logprecrot.scores)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(rotscores[:, 0], rotscores[:, 1], s=10, color="black")
for name, (x, y) in zip(stations, rotscores[:, :2], strict=True):
    ax.annotate(name, (x, y), fontsize=7, xytext=(2, 2), textcoords="offset points")
ax.axhline(0.0, color="grey", linestyle=":")
ax.axvline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Rotated component I score")
ax.set_ylabel("Rotated component II score")
fig

### Figure 7.4
Bivariate PCA of the 20 handwritten "fda" samples: for each of the first three
components, the mean (x(t), y(t)) trajectory with arrows to mean + 2 sd x harmonic,
showing how each component distorts the whole script.

In [ ]:
handwrit = fdt.datasets.load_handwriting()
hwbasis = fdt.BSpline(domain=(0.0, 2300.0), n_basis=105, order=6)
hwfit = fdt.smooth(handwrit.value, handwrit.t, basis=hwbasis, penalty=4)
print(f"handwriting: GCV lambda = {hwfit.lam:.3g}, df = {hwfit.df:.1f}")
hwpca = fdt.FPCA(n=3).fit(hwfit.fd)
hwvarprop = np.asarray(hwpca.varprop)
hwtime = np.linspace(0.0, 2300.0, 401)
hwmean = np.asarray(hwpca.mean_fd(hwtime))[:, 0, :]
hwharm = np.asarray(hwpca.harmonics(hwtime))
hwsd = np.sqrt(np.mean(np.asarray(hwpca.scores) ** 2, axis=0))
arrows = slice(0, None, 8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for j, ax in enumerate(axes):
    shifted = hwmean + 2.0 * hwsd[j] * hwharm[:, j, :]
    ax.plot(hwmean[:, 0], hwmean[:, 1], "k-", linewidth=1.5)
    ax.plot(shifted[:, 0], shifted[:, 1], color="tab:blue", linewidth=0.8)
    ax.quiver(
        hwmean[arrows, 0],
        hwmean[arrows, 1],
        (shifted - hwmean)[arrows, 0],
        (shifted - hwmean)[arrows, 1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=0.003,
        color="tab:red",
    )
    ax.set_aspect("equal")
    ax.set_title(f"PC {j + 1} ({100 * hwvarprop[j]:.1f}%)")
    ax.set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
fig.tight_layout()
fig

### Figure 7.5
First pair of canonical weight functions for temperature (solid) and log
precipitation (dashed), harmonic acceleration penalty lambda = 1e10 on both
(smaller values give rough weights and canonical correlations near one).

In [ ]:
weathercca = fdt.FCCA(n=3, lam1=1e10, lam2=1e10, penalty=harmonic).fit(tempfd, logprecfd)
canocor = np.asarray(weathercca.correlations)[:3]
print("canonical correlations:", np.round(canocor, 4))
dayfine = np.linspace(0.0, 365.0, 366)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dayfine, np.asarray(weathercca.weights1(dayfine))[:, 0], "k-", label="temperature")
ax.plot(dayfine, np.asarray(weathercca.weights2(dayfine))[:, 0], "k--", label="log precipitation")
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Day (July 1 to June 30)")
ax.set_ylabel("Canonical weight function")
ax.set_title(f"First canonical pair (r = {canocor[0]:.3f})")
ax.legend()
fig

### Figure 7.6
Scores of the 35 stations on the first canonical variable for log precipitation
against those for temperature, labelled by station.

In [ ]:
cscore1 = np.asarray(weathercca.scores1)[:, 0]
cscore2 = np.asarray(weathercca.scores2)[:, 0]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(cscore1, cscore2, s=10, color="black")
for name, x, y in zip(stations, cscore1, cscore2, strict=True):
    ax.annotate(name, (x, y), fontsize=7, xytext=(2, 2), textcoords="offset points")
ax.set_xlabel("Temperature canonical score")
ax.set_ylabel("Log precipitation canonical score")
fig

# Chapter 8: Registration: aligning features for samples of curves

Figures 8.1 to 8.7. The Berkeley growth data (first ten girls) are smoothed, their
height acceleration curves are aligned first by a landmark (the age of the pubertal
growth spurt) and then by continuous registration, and the variation is split into
amplitude and phase parts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["figure.autolayout"] = True

growth = fdt.datasets.load_growth()
age = growth.age
heights = growth.hgtf[:, :10]
height_basis = fdt.BSpline(domain=(1.0, 18.0), breaks=age, order=6)
height_fit = fdt.smooth(heights, age, basis=height_basis, lam=1e-1, penalty=4)
accel = height_fit.fd.derivative(2)

# Landmark: the age of the pubertal growth spurt (peak velocity, where the
# acceleration crosses zero from above).
spurt_grid = np.linspace(8.0, 16.0, 1601)
pgs_age = spurt_grid[np.argmax(height_fit.fd(spurt_grid, 1), axis=0)]

t_plot = np.linspace(3.0, 18.0, 401)
t_full = np.linspace(1.0, 18.0, 401)
landmark_fit = fdt.landmark_register(accel, pgs_age[:, None])
continuous_fit = fdt.register(
    landmark_fit.registered,
    landmark_fit.registered.mean(),
    warp_basis=fdt.BSpline(domain=(1.0, 18.0), n_basis=7),
    lam=1e-2,
)

### Figure 8.1
Height acceleration of the first ten girls of the Berkeley growth study with the
cross-sectional mean (heavy dashed). The mean understates the pubertal spurt because
the spurts happen at different ages. The book smooths with a monotone fit; a
D^4-penalised order-6 spline is used here (same curves up to small boundary effects).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_plot, accel(t_plot), color="0.45", linewidth=1)
ax.plot(t_plot, accel.mean()(t_plot)[:, 0], "k--", linewidth=2.5, label="cross-sectional mean")
ax.plot(pgs_age, np.zeros_like(pgs_age), "o", color="C3", label="PGS landmark")
ax.axhline(0.0, color="0.7", linewidth=0.5)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Height acceleration (cm/yr$^2$)")
ax.set_ylim(-4.0, 2.0)
ax.legend(loc="lower left")
fig

### Figure 8.2
A time-warping function h(t) for one girl (left, the diagonal is no warping) and the
effect of warping on her acceleration curve (right): the registered curve has its
pubertal spurt moved to the mean spurt age.

In [ ]:
girl = int(np.argmin(pgs_age))
h = landmark_fit.warp_values(t_full)[:, girl]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
ax1.plot(t_full, h, "C0", linewidth=2)
ax1.plot([1, 18], [1, 18], "k:", linewidth=1)
ax1.plot([pgs_age.mean()], [pgs_age[girl]], "o", color="C3")
ax1.set_xlabel("Clock time t (years)")
ax1.set_ylabel("Warped time h(t)")
ax1.set_title(f"Warping function, girl {girl + 1}")
ax2.plot(t_plot, accel[girl](t_plot)[:, 0], "C0--", label="unregistered")
ax2.plot(t_plot, landmark_fit.registered[girl](t_plot)[:, 0], "C0", label="registered")
ax2.plot(t_plot, landmark_fit.registered.mean()(t_plot)[:, 0], "k", linewidth=2, label="mean")
ax2.axhline(0.0, color="0.7", linewidth=0.5)
ax2.set_ylim(-4.0, 2.0)
ax2.set_xlabel("Age (years)")
ax2.set_ylabel("Acceleration (cm/yr$^2$)")
ax2.legend(loc="lower left")
fig.tight_layout()
fig

### Figure 8.3
The ten acceleration curves after landmark registration on the pubertal spurt, with
their mean (heavy line). The spurt now lines up and the mean keeps its depth.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_plot, landmark_fit.registered(t_plot), color="0.45", linewidth=1)
ax.plot(t_plot, landmark_fit.registered.mean()(t_plot)[:, 0], "k", linewidth=2.5)
ax.plot(t_plot, accel.mean()(t_plot)[:, 0], "k--", linewidth=1.5)
ax.axvline(pgs_age.mean(), color="C3", linestyle=":")
ax.axhline(0.0, color="0.7", linewidth=0.5)
ax.set_ylim(-4.0, 2.0)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Height acceleration (cm/yr$^2$)")
ax.set_title("Landmark-registered curves (solid mean) and unregistered mean (dashed)")
fig

### Figure 8.4
The landmark warping functions h_i(t) (left) and their deformations h_i(t) - t
(right). A girl with an early spurt has a warp below the diagonal around puberty.

In [ ]:
warps = landmark_fit.warp_values(t_full)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
ax1.plot(t_full, warps)
ax1.plot([1, 18], [1, 18], "k:", linewidth=1)
ax1.set_xlabel("Age (years)")
ax1.set_ylabel("h(t)")
ax1.set_title("Landmark warping functions")
ax2.plot(t_full, warps - t_full[:, None])
ax2.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax2.set_xlabel("Age (years)")
ax2.set_ylabel("h(t) - t")
ax2.set_title("Deformations")
fig.tight_layout()
fig

### Figure 8.5
Continuous registration (minimum-eigenvalue criterion) of the landmark-registered
curves to their mean, with a 7-function cubic B-spline basis for W and lambda = 0.01.
The mean of the registered curves is the heavy line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_plot, continuous_fit.registered(t_plot), color="0.45", linewidth=1)
ax.plot(t_plot, continuous_fit.registered.mean()(t_plot)[:, 0], "k", linewidth=2.5)
ax.axhline(0.0, color="0.7", linewidth=0.5)
ax.set_ylim(-4.0, 2.0)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Height acceleration (cm/yr$^2$)")
ax.set_title("Continuously registered acceleration curves")
fig

### Figure 8.6
Warping functions of the continuous registration (left) and their deformations
h(t) - t (right). They are small because the landmark step already removed most of
the phase variation.

In [ ]:
warps_c = continuous_fit.warp_values(t_full)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
ax1.plot(t_full, warps_c)
ax1.plot([1, 18], [1, 18], "k:", linewidth=1)
ax1.set_xlabel("Age (years)")
ax1.set_ylabel("h(t)")
ax1.set_title("Continuous warping functions")
ax2.plot(t_full, warps_c - t_full[:, None])
ax2.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax2.set_xlabel("Age (years)")
ax2.set_ylabel("h(t) - t")
ax2.set_title("Deformations")
fig.tight_layout()
fig

### Figure 8.7
Amplitude and phase decomposition (Kneip and Ramsay 2008): the three cross-sectional
means (unregistered, landmark, landmark + continuous) and, per registration, the
share of the variation that was due to phase (RSQ). The continuous step starts from the
landmark-registered curves, so its phase part is small.

In [ ]:
landmark_parts = landmark_fit.decompose(domain=(3.0, 18.0))
continuous_parts = continuous_fit.decompose(domain=(3.0, 18.0))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [2, 1]})
ax1.plot(t_plot, accel.mean()(t_plot)[:, 0], "k--", label="unregistered")
ax1.plot(t_plot, landmark_fit.registered.mean()(t_plot)[:, 0], "C0", label="landmark")
ax1.plot(t_plot, continuous_fit.registered.mean()(t_plot)[:, 0], "C3", label="continuous")
ax1.axhline(0.0, color="0.7", linewidth=0.5)
ax1.set_xlabel("Age (years)")
ax1.set_ylabel("Mean acceleration (cm/yr$^2$)")
ax1.legend(loc="lower left")
names = ["landmark", "continuous\n(after landmark)"]
position = np.arange(2)
amp = [landmark_parts.amp_mse, continuous_parts.amp_mse]
phase = [landmark_parts.phase_mse, continuous_parts.phase_mse]
ax2.bar(position - 0.2, amp, width=0.4, color="C0", label="amplitude MSE")
ax2.bar(position + 0.2, phase, width=0.4, color="C1", label="phase MSE")
for k, parts in enumerate([landmark_parts, continuous_parts]):
    ax2.text(k, max(amp[k], phase[k]) * 1.05, f"RSQ={parts.rsq:.2f}", ha="center")
ax2.set_xticks(position, names)
ax2.axhline(0.0, color="k", linewidth=0.5)
ax2.set_ylabel("Mean squared error")
ax2.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig

# Chapter 9: Functional linear models for scalar responses

Figures 9.1 to 9.7. The log10 total annual precipitation at the 35 Canadian weather
stations is predicted from each station's daily temperature profile:
y_i = alpha + integral of x_i(t) beta(t) dt + e_i.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["figure.autolayout"] = True

weather = fdt.datasets.load_canadian_weather()
day = weather.t - 0.5
annual_precip = np.log10(weather.precip.sum(axis=0))
harmonic = fdt.LDO.harmonic(365.0)
temp_basis = fdt.Fourier(domain=(0.0, 365.0), n_basis=65)
temp = fdt.smooth(weather.temp, day, basis=temp_basis, lam=1e-2, penalty=harmonic).fd
covariates = {"const": 1.0, "temp": temp}
t_grid = np.linspace(0.0, 365.0, 366)
month_ticks = np.cumsum([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30])
month_names = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

### Figure 9.1
Estimated regression function beta(t) when beta is expanded in only five Fourier basis
functions and no roughness penalty is used. The low dimension alone keeps it smooth.

In [ ]:
model_low = fdt.fregress(annual_precip, covariates, beta={"temp": fdt.Fourier((0.0, 365.0), 5)})
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_grid, model_low.beta[1](t_grid)[:, 0], "C0", linewidth=2)
ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("beta(t) for temperature")
ax.set_title(f"Five Fourier basis functions, df = {model_low.df:.0f}")
fig

### Figure 9.2
Leave-one-out cross-validation score (sum of squared prediction errors) as a function
of log10 lambda, for beta in 35 Fourier functions with a harmonic acceleration penalty.

In [ ]:
beta_basis = fdt.Fourier((0.0, 365.0), 35)
log_lambdas = np.arange(6.0, 15.01, 0.5)
cv_scores = []
for log_lambda in log_lambdas:
    trial = fdt.fregress(
        annual_precip, covariates, beta={"temp": (beta_basis, 10.0**log_lambda, harmonic)}
    )
    cv_scores.append(trial.cv().sse)
best_log_lambda = float(log_lambdas[int(np.argmin(cv_scores))])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(log_lambdas, cv_scores, "o-", color="C0")
ax.axvline(best_log_lambda, color="C3", linestyle=":")
ax.set_xlabel("log10 smoothing parameter lambda")
ax.set_ylabel("Cross-validation score")
ax.set_title(f"Minimum at log10 lambda = {best_log_lambda:.1f}")
fig

### Figure 9.3
The regression function beta(t) estimated with the roughness penalty at the
cross-validated lambda: temperature in late autumn and winter predicts precipitation.

In [ ]:
model = fdt.fregress(
    annual_precip, covariates, beta={"temp": (beta_basis, 10.0**best_log_lambda, harmonic)}
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_grid, model.beta[1](t_grid)[:, 0], "C0", linewidth=2)
ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("beta(t) for temperature")
ax.set_title(f"Harmonic acceleration penalty, df = {model.df:.2f}")
fig

### Figure 9.4
Observed log10 annual precipitation against the values fitted by the penalised model.
The diagonal is perfect prediction; a few coastal stations are the largest misses.

In [ ]:
fitted = np.asarray(model.fitted)
r_squared = 1.0 - np.sum((annual_precip - fitted) ** 2) / np.sum(
    (annual_precip - annual_precip.mean()) ** 2
)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(fitted, annual_precip, "o", color="C0")
limits = [min(fitted.min(), annual_precip.min()), max(fitted.max(), annual_precip.max())]
ax.plot(limits, limits, "k--", linewidth=1)
for k in np.argsort(np.abs(annual_precip - fitted))[-3:]:
    ax.annotate(weather.stations[k], (fitted[k], annual_precip[k]), fontsize=8)
ax.set_xlabel("Fitted log10 annual precipitation")
ax.set_ylabel("Observed log10 annual precipitation")
ax.set_title(f"R$^2$ = {r_squared:.2f}")
fig

### Figure 9.5
The penalised estimate of beta(t) with pointwise 95% confidence limits (plus and minus
two standard errors), from the error variance SSE / (n - df).

In [ ]:
stderr = model.stderr()
beta_values = model.beta[1](t_grid)[:, 0]
beta_se = stderr.beta[1](t_grid)[:, 0]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_grid, beta_values, "C0", linewidth=2)
ax.plot(t_grid, beta_values + 2.0 * beta_se, "C0--", linewidth=1)
ax.plot(t_grid, beta_values - 2.0 * beta_se, "C0--", linewidth=1)
ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("beta(t) for temperature")
fig

### Figure 9.6
Functional principal components regression: log precipitation is regressed on the
scores of the first four (lightly smoothed) temperature harmonics, and
beta(t) = sum_k b_k xi_k(t) is shown with plus and minus two standard errors.

In [ ]:
pca = fdt.FPCA(n=4, lam=1e5, penalty=harmonic).fit(temp)
scores = np.asarray(pca.scores)
score_terms = {"const": 1.0} | {f"pc{k + 1}": scores[:, k] for k in range(4)}
pcr = fdt.fregress(annual_precip, score_terms)
score_coefs = np.asarray(pcr.coefficients)[1:]
score_cov = np.asarray(pcr.stderr().cov)[1:, 1:]
harmonic_values = np.asarray(pca.harmonics(t_grid))
pcr_beta = harmonic_values @ score_coefs
pcr_se = np.sqrt(np.einsum("tk,kl,tl->t", harmonic_values, score_cov, harmonic_values))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_grid, pcr_beta, "C2", linewidth=2, label="principal components regression")
ax.plot(t_grid, pcr_beta + 2.0 * pcr_se, "C2--", linewidth=1)
ax.plot(t_grid, pcr_beta - 2.0 * pcr_se, "C2--", linewidth=1)
ax.plot(t_grid, beta_values, "C0", linewidth=1, label="roughness penalty")
ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("beta(t) for temperature")
ax.legend(loc="upper left", fontsize=8)
fig

### Figure 9.7
Permutation test of no relation between temperature and precipitation: histogram of
the F statistic over 200 random permutations of the responses, the observed F (solid)
and the 95% permutation quantile (dashed).

In [ ]:
test = fdt.stats.f_test(
    annual_precip,
    [np.ones(temp.n_curves), temp],
    basis=[None, beta_basis],
    lam=[0.0, 10.0**best_log_lambda],
    penalty=[2, harmonic],
    n_perm=200,
    random_state=2009,
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(test.null, bins=25, color="0.7", edgecolor="0.4")
ax.axvline(test.statistic, color="C3", linewidth=2, label=f"observed F = {test.statistic:.2f}")
ax.axvline(test.critical_value, color="k", linestyle="--", label="95% permutation quantile")
ax.set_xlabel("F statistic")
ax.set_ylabel("Frequency")
ax.set_title(f"Permutation p-value = {test.pvalue:.3f}")
ax.legend(fontsize=8)
fig

# Chapter 10: Linear models for functional responses

Figures 10.1 to 10.8. A functional analysis of variance of the Canadian temperature
curves by climate region, the concurrent model of log precipitation on temperature,
and the concurrent model of knee angle on hip angle in the gait data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["figure.autolayout"] = True

weather = fdt.datasets.load_canadian_weather()
day = weather.t - 0.5
n_stations = len(weather.stations)
harmonic = fdt.LDO.harmonic(365.0)
day_basis = fdt.Fourier(domain=(0.0, 365.0), n_basis=65)
temp = fdt.smooth(weather.temp, day, basis=day_basis, lam=1e-2, penalty=harmonic).fd
regions = sorted(set(weather.region))
region_dummy = {
    name: np.array([1.0 if r == name else 0.0 for r in weather.region]) for name in regions
}
t_grid = np.linspace(0.0, 365.0, 366)
month_ticks = np.cumsum([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30])
month_names = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
region_colors = dict(zip(regions, ["C0", "C1", "C2", "C3"], strict=True))

### Figure 10.1
Functional ANOVA of temperature by climate region: the Canada-wide mean function
and the four regional effects. The effects are made to sum to zero by appending one
pseudo-observation (a zero curve with every region indicator on), as in the book.

In [ ]:
anova_y = fdt.FData(
    np.hstack([np.asarray(temp.coefs), np.zeros((day_basis.n_basis, 1))]), day_basis
)
anova_x = {"const": np.r_[np.ones(n_stations), 0.0]}
anova_x |= {name: np.r_[region_dummy[name], 1.0] for name in regions}
anova = fdt.fregress(anova_y, anova_x, beta=fdt.Fourier((0.0, 365.0), 11))
fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True)
for ax, name, beta in zip(axes.ravel(), anova.names, anova.beta, strict=False):
    ax.plot(t_grid, beta(t_grid)[:, 0], linewidth=2, color=region_colors.get(name, "k"))
    ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
    ax.set_title("Canada mean" if name == "const" else f"{name} effect")
    ax.set_xticks(month_ticks, month_names)
    ax.set_ylabel("deg C")
axes.ravel()[-1].axis("off")
fig

### Figure 10.2
Temperature curves of the stations (thin) coloured by region, with the fitted mean
curve of each region (thick): the prediction of the ANOVA model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, region in enumerate(weather.region):
    ax.plot(t_grid, temp[k](t_grid)[:, 0], color=region_colors[region], linewidth=0.5, alpha=0.6)
for name in regions:
    new_x = {"const": 1.0} | {other: float(other == name) for other in regions}
    prediction = anova.predict(new_x)
    ax.plot(t_grid, prediction(t_grid)[:, 0], color=region_colors[name], linewidth=3, label=name)
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("Mean temperature (deg C)")
ax.legend(fontsize=8)
fig

### Figure 10.3
Permutation F test for a region effect: the observed pointwise F statistic (solid),
the pointwise 95% permutation quantiles (dotted) and the 95% quantile of the maximum
of F (dashed), from 200 permutations (treatment coding, Arctic as baseline).

In [ ]:
region_test = fdt.stats.f_test(
    temp,
    [np.ones(n_stations)] + [region_dummy[name] for name in regions[1:]],
    basis=fdt.Fourier((0.0, 365.0), 11),
    n_perm=200,
    random_state=2009,
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(region_test.t, region_test.pointwise, "C0", linewidth=2, label="observed F(t)")
ax.plot(region_test.t, region_test.pointwise_critical_value, "k:", label="pointwise 0.95")
ax.axhline(region_test.critical_value, color="k", linestyle="--", label="maximum 0.95")
ax.set_xticks(month_ticks, month_names)
ax.set_xlabel("Day")
ax.set_ylabel("F statistic")
ax.set_title(f"Permutation p-value of max F = {region_test.pvalue:.3f}")
ax.legend(fontsize=8)
fig

### Figure 10.4
The concurrent model log10 precipitation(t) = beta0(t) + beta1(t) temperature(t):
intercept (left) and temperature coefficient (right) with plus and minus two pointwise
standard errors, from the residual covariance over days and the smoothing map.

In [ ]:
log_precip = fdt.smooth(weather.log10precip, day, basis=day_basis, lam=1e5, penalty=harmonic)
concurrent = fdt.fregress(
    log_precip, {"const": 1.0, "temp": temp}, beta=fdt.Fourier((0.0, 365.0), 11)
)
residuals = weather.log10precip - np.asarray(concurrent.fitted(day))
concurrent_se = concurrent.stderr(sigma_e=np.cov(residuals))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name, beta, se in zip(
    axes, ["Intercept", "Temperature coefficient"], concurrent.beta, concurrent_se.beta, strict=True
):
    values = beta(t_grid)[:, 0]
    width = 2.0 * se(t_grid)[:, 0]
    ax.plot(t_grid, values, "C0", linewidth=2)
    ax.plot(t_grid, values + width, "C0--", linewidth=1)
    ax.plot(t_grid, values - width, "C0--", linewidth=1)
    ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
    ax.set_xticks(month_ticks, month_names)
    ax.set_title(name)
fig

### Figure 10.5
Leave-one-station-out cross-validated error sum of squares of the concurrent model as
a function of the harmonic acceleration penalty on both coefficient functions. A
moderate penalty improves the out-of-sample fit slightly; a heavy one hurts it.

In [ ]:
log_lambdas = np.arange(4.0, 14.01, 1.0)
cv_errors = [
    fdt.fregress(
        log_precip,
        {"const": 1.0, "temp": temp},
        beta=fdt.Fourier((0.0, 365.0), 11),
        lam=10.0**value,
        penalty=harmonic,
    )
    .cv()
    .sse
    for value in log_lambdas
]
fig, ax = plt.subplots(figsize=(7, 4))
best = int(np.argmin(cv_errors))
ax.plot(log_lambdas, cv_errors, "o-", color="C0")
ax.axvline(log_lambdas[best], color="C3", linestyle=":")
ax.set_title(f"Minimum at log10 lambda = {log_lambdas[best]:.0f}")
ax.set_xlabel("log10 lambda (harmonic acceleration penalty)")
ax.set_ylabel("Cross-validated SSE")
fig

### Figure 10.6
The gait data: hip angle (left) and knee angle (right) of 39 boys over one gait
cycle, smoothed with a 21-function Fourier basis (time rescaled to [0, 1]).

In [ ]:
gait = fdt.datasets.load_gait()
gait_basis = fdt.Fourier(domain=(0.0, 1.0), n_basis=21)
gait_penalty = fdt.LDO.harmonic(1.0)
hip = fdt.smooth(gait.hip_angle, gait.t, basis=gait_basis, lam=1e-11, penalty=gait_penalty).fd
knee_fit = fdt.smooth(gait.knee_angle, gait.t, basis=gait_basis, lam=1e-11, penalty=gait_penalty)
knee = knee_fit.fd
cycle = np.linspace(0.0, 1.0, 201)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(cycle, hip(cycle), linewidth=0.7)
axes[0].set_title("Hip angle")
axes[1].plot(cycle, knee(cycle), linewidth=0.7)
axes[1].set_title("Knee angle")
for ax in axes:
    ax.set_xlabel("Time (proportion of gait cycle)")
    ax.set_ylabel("degrees")
fig

### Figure 10.7
Concurrent model knee(t) = beta0(t) + beta1(t) hip(t): the intercept, the hip
coefficient with plus and minus two standard errors, and the squared multiple
correlation R^2(t) along the cycle.

In [ ]:
gait_model = fdt.fregress(
    knee_fit,
    {"const": 1.0, "hip": hip},
    beta=fdt.Fourier((0.0, 1.0), 21),
    lam=1e-8,
    penalty=gait_penalty,
)
gait_residuals = gait.knee_angle - np.asarray(gait_model.fitted(gait.t))
gait_se = gait_model.stderr(sigma_e=np.cov(gait_residuals))
knee_values = knee(cycle)
fit_values = gait_model.fitted(cycle)
r_squared = 1.0 - np.sum((knee_values - fit_values) ** 2, axis=1) / np.sum(
    (knee_values - knee_values.mean(axis=1, keepdims=True)) ** 2, axis=1
)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, title, beta, se in zip(
    axes[:2], ["Intercept", "Hip coefficient"], gait_model.beta, gait_se.beta, strict=True
):
    values = beta(cycle)[:, 0]
    width = 2.0 * se(cycle)[:, 0]
    ax.plot(cycle, values, "C0", linewidth=2)
    ax.plot(cycle, values + width, "C0--", linewidth=1)
    ax.plot(cycle, values - width, "C0--", linewidth=1)
    ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Time (proportion of gait cycle)")
axes[2].plot(cycle, r_squared, "C2", linewidth=2)
axes[2].set_ylim(0.0, 1.0)
axes[2].set_title("Squared multiple correlation R$^2$(t)")
axes[2].set_xlabel("Time (proportion of gait cycle)")
fig

### Figure 10.8
Observed knee angle (solid) and the knee angle predicted from the hip angle by the
concurrent model (dashed) for four boys.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, boy in zip(axes.ravel(), [0, 9, 19, 29], strict=True):
    ax.plot(cycle, knee_values[:, boy], "C0", linewidth=2, label="observed")
    ax.plot(cycle, fit_values[:, boy], "C3--", linewidth=2, label="predicted")
    ax.plot(gait.t, gait.knee_angle[:, boy], "o", color="C0", markersize=3)
    ax.set_title(f"Boy {boy + 1}")
axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Time (proportion of gait cycle)")
for ax in axes[:, 0]:
    ax.set_ylabel("Knee angle (degrees)")
fig

# Chapter 11: Functional models and dynamics

Figures 11.1 to 11.9. Principal differential analysis (PDA) of the lip movement data,
a coupled third-order model of handwriting, a first-order forced model of the oil
refinery data and gradient matching for a simulated stirred-tank chemical reactor.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

import fdatools as fdt

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["figure.autolayout"] = True

lip = fdt.datasets.load_lip()
lip_domain = (float(lip.t[0]), float(lip.t[-1]))
lip_basis = fdt.BSpline(domain=lip_domain, breaks=lip.t, order=6)
lip_fd = fdt.smooth(lip.value, lip.t, basis=lip_basis, lam=1e-12, penalty=4).fd
lip_grid = np.linspace(lip_domain[0], lip_domain[1], 201)
lip_pda = fdt.PDA(order=2, weight_basis=fdt.BSpline(domain=lip_domain, n_basis=7)).fit(lip_fd)

### Figure 11.1
Lower lip position during 20 repetitions of the syllable "bob" (left) and the lip
acceleration (right), from an order-6 spline with a D^4 roughness penalty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lip_grid, lip_fd(lip_grid), linewidth=0.8)
axes[0].set_ylabel("Lip position (mm)")
axes[1].plot(lip_grid, lip_fd(lip_grid, 2), linewidth=0.8)
axes[1].set_ylabel("Lip acceleration (mm/s$^2$)")
for ax in axes:
    ax.set_xlabel("Time (s)")
fig

### Figure 11.2
Phase-plane plot of the mean lip curve: acceleration against velocity. The loops
show the exchange between kinetic and potential energy as the lip opens and closes.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
fdt.phase_plane(
    lip_fd.mean(),
    lip_grid,
    labels={0.05: "0.05 s", 0.15: "0.15 s", 0.25: "0.25 s", 0.33: "0.33 s"},
    ax=ax,
    color="C0",
    linewidth=2,
)
ax.axhline(0.0, color="0.6", linewidth=0.5)
ax.axvline(0.0, color="0.6", linewidth=0.5)
ax.set_xlabel("Velocity (mm/s)")
ax.set_ylabel("Acceleration (mm/s$^2$)")
fig

### Figure 11.3
The PDA weight functions of D^2 x + beta1(t) Dx + beta0(t) x = 0 estimated from the
lip curves (7 cubic B-spline functions each): the stiffness beta0 and damping beta1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lip_grid, lip_pda.weights_[0](lip_grid)[:, 0], "C0", linewidth=2)
axes[0].set_title("beta0(t)")
axes[1].plot(lip_grid, lip_pda.weights_[1](lip_grid)[:, 0], "C1", linewidth=2)
axes[1].set_title("beta1(t)")
for ax in axes:
    ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
    ax.set_xlabel("Time (s)")
fig

### Figure 11.4
Stability diagram of the lip model: the path (beta1(t), beta0(t)) against the parabola
beta0 = beta1^2 / 4. Above it the system oscillates; negative beta1 amplifies.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
lip_pda.plot_overlay(
    ax=ax, labels={0.0: "0 s", 0.1: "0.1 s", 0.2: "0.2 s", 0.3: "0.3 s"}, color="C0", linewidth=2
)
ax.set_title("PDA stability diagram, lip data")
fig

### Figure 11.5
Solutions of the estimated equation L x = 0 started from each curve's initial
position and velocity (dashed) compared with the smoothed lip curves (solid), for
four repetitions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, rep in zip(axes.ravel(), [0, 5, 10, 15], strict=True):
    start = [lip_fd(lip_grid[:1])[0, rep], lip_fd(lip_grid[:1], 1)[0, rep]]
    ax.plot(lip_grid, lip_fd(lip_grid)[:, rep], "C0", linewidth=2, label="data")
    ax.plot(lip_grid, lip_pda.solve(lip_grid, start), "C3--", linewidth=2, label="L x = 0")
    ax.set_title(f"Repetition {rep + 1}")
axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Lip position (mm)")
fig

### Figure 11.6
A coupled third-order PDA of handwriting, D^3 x_i = -sum_k sum_j beta_ikj(t) D^j x_k,
fitted to 20 samples and solved from the mean curve's initial state. The book uses
Chinese script, which fdatools does not ship; the cursive "fda" data stand in for it.

In [ ]:
writing = fdt.datasets.load_handwriting()
pen_domain = (0.0, 2300.0)
pen_basis = fdt.BSpline(domain=pen_domain, n_basis=105, order=6)
pen_fit = fdt.smooth(writing.value[::2], writing.t[::2], basis=pen_basis, lam=1e2, penalty=4)
pen_pda = fdt.PDA(
    order=3, weight_basis=fdt.BSpline(domain=pen_domain, n_basis=43), n_grid=1001
).fit(pen_fit.fd)
pen_grid = np.linspace(pen_domain[0], pen_domain[1], 461)
pen_mean = pen_fit.fd.mean()
mean_values = pen_mean(pen_grid)[:, 0, :]
pen_start = np.array([[pen_mean(pen_grid[:1], j)[0, 0, v] for j in range(3)] for v in range(2)])
pen_solution = pen_pda.solve(pen_grid, pen_start)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), gridspec_kw={"width_ratios": [1.4, 1, 1]})
axes[0].plot(mean_values[:, 0], mean_values[:, 1], "C0", linewidth=2, label="mean script")
axes[0].plot(pen_solution[:, 0], pen_solution[:, 1], "C3--", linewidth=1.5, label="L x = 0")
axes[0].set_aspect("equal")
axes[0].legend(fontsize=8)
axes[0].set_title("Script")
for k, name in enumerate(["X", "Y"]):
    axes[k + 1].plot(pen_grid, mean_values[:, k], "C0", linewidth=2)
    axes[k + 1].plot(pen_grid, pen_solution[:, k], "C3--", linewidth=1.5)
    axes[k + 1].set_title(f"{name} coordinate")
    axes[k + 1].set_xlabel("Time (ms)")
fig

### Figure 11.7
The refinery data: a step drop in reflux flow (bottom) and the tray 47 level (top),
with the solution of the forced first-order model Dx = -beta x + alpha u. The
constant coefficients come from a concurrent regression of Dx on x and u with
constant coefficient bases (PDA forcing functions are not in fdatools).

In [ ]:
refinery = fdt.datasets.load_refinery()
ref_domain = (0.0, 193.0)
reflux_fd = fdt.smooth(
    refinery.reflux,
    refinery.time,
    basis=fdt.BSpline(ref_domain, order=1, breaks=[0.0, 67.0, 193.0]),
    lam=0.0,
).fd
tray_breaks = np.unique(np.r_[np.linspace(0.0, 67.0, 5), np.linspace(70.0, 193.0, 30)])
tray_basis = fdt.BSpline(ref_domain, breaks=tray_breaks, order=4)
tray_fd = fdt.smooth(refinery.tray47, refinery.time, basis=tray_basis, lam=1.0).fd
forced = fdt.fregress(
    tray_fd.derivative(), {"tray": tray_fd, "reflux": reflux_fd}, beta=fdt.Constant(ref_domain)
)
beta_hat = -float(np.asarray(forced.beta[0].coefs)[0, 0])
alpha_hat = float(np.asarray(forced.beta[1].coefs)[0, 0])
tray_solution = solve_ivp(
    lambda s, x: [-beta_hat * x[0] + alpha_hat * float(reflux_fd(np.array([s]))[0, 0])],
    ref_domain,
    [float(tray_fd(np.array([0.0]))[0, 0])],
    t_eval=refinery.time,
    max_step=1.0,
).y[0]
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(refinery.time, refinery.tray47, "o", color="0.5", markersize=3, label="data")
axes[0].plot(refinery.time, tray_solution, "C3", linewidth=2, label="fitted ODE")
axes[0].set_ylabel("Tray 47 level")
axes[0].set_title(f"beta = {beta_hat:.4f}, alpha = {alpha_hat:.3f}")
axes[0].legend(fontsize=8)
axes[1].plot(refinery.time, refinery.reflux, "o", color="0.5", markersize=3)
axes[1].plot(refinery.time, reflux_fd(refinery.time)[:, 0], "C0", linewidth=2)
axes[1].set_ylabel("Reflux flow")
axes[1].set_xlabel("Time (min)")
fig

### Figure 11.8
A continuously stirred tank reactor: input flow and input temperature (bottom) drive
output concentration C and temperature T (top). The book's CSTR measurements are not
in fdatools, so a dimensionless CSTR model is simulated (fixed seed) and noise is added.

In [ ]:
def flow(s):
    return 1.0 - 0.3 * (16.0 <= s < 32.0) + 0.25 * (40.0 <= s < 52.0)


def inlet_temp(s):
    return 1.0 + 0.15 * (8.0 <= s < 24.0) - 0.1 * (48.0 <= s < 60.0)


activation, inlet_conc, coolant = 6.0, 1.0, 0.9
true_params = (0.8, 0.6, 1.0)  # rate constant, heat of reaction, heat transfer


def reactor(s, state, rate, heat, transfer):
    conc, temp = state
    k = rate * np.exp(-activation * (1.0 / temp - 1.0))
    return [
        flow(s) * (inlet_conc - conc) - k * conc,
        flow(s) * (inlet_temp(s) - temp) + heat * k * conc - transfer * (temp - coolant),
    ]


cstr_time = np.linspace(0.0, 64.0, 257)
truth = solve_ivp(
    reactor, (0.0, 64.0), [0.5, 1.0], t_eval=cstr_time, args=true_params, max_step=0.1
).y.T
rng = np.random.default_rng(2009)
observed = truth + rng.normal(scale=[0.01, 0.005], size=truth.shape)
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
axes[0, 0].plot(cstr_time, observed[:, 0], ".", color="0.5", markersize=3)
axes[0, 0].plot(cstr_time, truth[:, 0], "C0", linewidth=1.5)
axes[0, 0].set_title("Output concentration C")
axes[0, 1].plot(cstr_time, observed[:, 1], ".", color="0.5", markersize=3)
axes[0, 1].plot(cstr_time, truth[:, 1], "C3", linewidth=1.5)
axes[0, 1].set_title("Output temperature T")
axes[1, 0].plot(cstr_time, [flow(s) for s in cstr_time], "k")
axes[1, 0].set_title("Input flow")
axes[1, 1].plot(cstr_time, [inlet_temp(s) for s in cstr_time], "k")
axes[1, 1].set_title("Input temperature")
for ax in axes[1]:
    ax.set_xlabel("Time (min)")
fig

### Figure 11.9
Gradient matching for the reactor: the noisy outputs are smoothed, the smooth curves
and their derivatives give the three reactor constants by least squares, and the
model is re-solved with them (dashed) over the data. fdatools has no nonlinear ODE
estimator (parameter cascading), so this is built from smooth() and derivatives.

In [ ]:
cstr_fd = fdt.smooth(observed, cstr_time, basis=fdt.BSpline((0.0, 64.0), n_basis=70), lam=1e-2).fd
inner = np.linspace(1.0, 63.0, 621)
conc_s, temp_s = cstr_fd(inner).T
dconc_s, dtemp_s = cstr_fd(inner, 1).T
flows = np.array([flow(s) for s in inner])
inlets = np.array([inlet_temp(s) for s in inner])
reaction = np.exp(-activation * (1.0 / temp_s - 1.0)) * conc_s
# dC + q (C - Cin) = -rate * e(T) C
rate_hat = float(np.linalg.lstsq(-reaction[:, None], dconc_s + flows * (conc_s - inlet_conc))[0][0])
# dT - q (Tin - T) = heat * rate * e(T) C - transfer (T - Tc)
design = np.column_stack([rate_hat * reaction, -(temp_s - coolant)])
heat_hat, transfer_hat = np.linalg.lstsq(design, dtemp_s - flows * (inlets - temp_s))[0]
refit = solve_ivp(
    reactor,
    (0.0, 64.0),
    list(observed[0]),
    t_eval=cstr_time,
    args=(rate_hat, heat_hat, transfer_hat),
    max_step=0.1,
).y.T
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k, name in enumerate(["Concentration C", "Temperature T"]):
    axes[k].plot(cstr_time, observed[:, k], ".", color="0.5", markersize=3, label="data")
    axes[k].plot(cstr_time, refit[:, k], "C3--", linewidth=2, label="re-solved ODE")
    axes[k].set_title(name)
    axes[k].set_xlabel("Time (min)")
axes[0].legend(fontsize=8)
fig.suptitle(
    f"estimates (truth): rate {rate_hat:.3f} ({true_params[0]}), "
    f"heat {heat_hat:.3f} ({true_params[1]}), transfer {transfer_hat:.3f} ({true_params[2]})"
)
fig